# GSoC 2026: ML4SCI (EXXA Gen Test)
## Denoising Astronomical Observations of Protoplanetary Disks
**Applicant:** Arya Wadhwa

This notebook demonstrates the end-to-end execution of a custom Convolutional Autoencoder and Latent-Space Clustering pipeline designed specifically for ALMA interferometry data (`.fits` cubes). 

### Pipeline Overview
1. **`alma_dataset.py`**: Custom PyTorch dataset. Extracs the Stokes I continuum layer, handles `arcsinh` stretching, and parses physical beam metadata (CDELT) from FITS headers.
2. **`alma_autoencoder.py`**: Fully convolutional bottleneck architecture minimizing a combined L1 + MS-SSIM reconstruction loss.
3. **`alma_clustering.py`**: UMAP dimensionality reduction on the 256-D latent space, K-Means clustering, and automated morphological inference (identifying gaps, rings, compact disks) via azimuthal radial profiles.

### 1. Setup Environment

In [ ]:
!pip install torch torchvision astropy numpy matplotlib scikit-learn umap-learn tqdm scipy

### 2. Prepare Data
*Note: Ensure you have uploaded the official ML4SCI EXXA `.fits` files to the `data/fits/` directory in the Colab file explorer, and uploaded the three `.py` scripts.*

In [ ]:
import os
os.makedirs("data/fits", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Directories created. Please upload FITS cubes to data/fits/")

### 3. Train the ALMA Convolutional Autoencoder

In [ ]:
from alma_dataset import build_dataloader
from alma_autoencoder import ALMAAutoencoder, train, TrainingConfig

# Construct DataLoader with augmented transformations to prevent overfitting 
# on the small public ALMA datasets.
loader = build_dataloader(fits_dir="data/fits", batch_size=8, augment=True, num_workers=2)

config = TrainingConfig(
    epochs=500,
    lr=1e-3,
    weight_decay=1e-5,
    checkpoint_dir="checkpoints"
)

model = ALMAAutoencoder(in_channels=1, latent_dim=256)
history = train(model, loader, config=config)

### 4. Cluster Latent Space & Generate Morphological Analytics

In [ ]:
from alma_clustering import run_pipeline, ClusteringConfig

# Execute clustering on the best checkpoint generated during training.
clustering_cfg = ClusteringConfig(
    n_clusters=6,
    reducer="umap",
    clusterer="kmeans",
    output_dir="results"
)

result = run_pipeline(
    model_checkpoint="checkpoints/best.pth", 
    fits_dir="data/fits", 
    config=clustering_cfg
)

print("\n--- Analysis Complete ---")
print("Images have been saved to the results/ directory: embedding, gallery, and radial profiles.")